<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.5-multimodal-models/notebooks/exercises-4_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.5: Multimodal Models: Gemini Vision

*Module 4 · Image, Vision & Multimodal AI*

> Send an image + a question. The AI answers. Send a document screenshot. The AI extracts every field. Send a video. The AI summarizes it. Welcome to multimodal AI — one model that handles text, images, audio, and video together.

---

## About this notebook

This notebook contains the **10 runnable code examples** from the published lesson `lesson-4.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. `setup.py`
2. Step 1 — Image Understanding — Ask Questions About Any Photo — `part1a_basic_vision.py`
3. Step 1 — Image Understanding — Ask Questions About Any Photo — `part1b_visual_qa.py`
4. Step 2 — Document AI — Extract Data from Photos of Documents — `part2_document_extraction.py`
5. Step 2 — Document AI — Extract Data from Photos of Documents — `part2_batch_receipts.py`
6. Step 3 — Multi-Image Reasoning — Compare, Contrast, Analyze — `part3_multi_image.py`
7. Step 4 — Video Understanding — Watch and Analyze Videos — `part4_video_understanding.py`
8. Step 5 — Project: Smart Product Catalog Generator — `project_product_catalog.py`
9. Step 5 — Project: Smart Product Catalog Generator — `llava_local.py`
10. Step 6 — How VLMs Work — Vision Encoder + Projector + LLM — `vlm_architecture.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers torch pillow pydantic accelerate


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


**`setup.py`** · _Block 1 of 10_

pip install google-generativeai pillow


In [ ]:
# pip install google-generativeai pillow
import google.generativeai as genai
from PIL import Image
import os, json

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")


### Step 1 · Image Understanding — Ask Questions About Any Photo

Send an image + a question. The AI sees and answers.

**`part1a_basic_vision.py`** · _Block 2 of 10_

IMAGE UNDERSTANDING: The simplest multimodal call — Send an image + text → get an intelligent answer


In [ ]:
# =============================================
# IMAGE UNDERSTANDING: The simplest multimodal call
# Send an image + text → get an intelligent answer
# =============================================

from PIL import Image

# Load any image
image = Image.open("street_food.jpg")

# Ask a question about it
response = model.generate_content([
    "What street food items can you see in this image? "
    "For each item, identify the name, estimated price range in INR, "
    "and one interesting fact about it.",
    image
])

print(response.text)

# The AI might respond:
# "I can see several street food items:
# 1. Pani Puri - ₹20-40 per plate - Also called Golgappa in North India
# 2. Bhel Puri - ₹30-50 per plate - Originated in Mumbai's Chowpatty Beach
# 3. Vada Pav - ₹15-25 each - Often called India's burger"


**`part1b_visual_qa.py`** · _Block 3 of 10_

VISUAL Q&A: Ask DIFFERENT questions about the — SAME image — get different expert answers.


In [ ]:
# =============================================
# VISUAL Q&A: Ask DIFFERENT questions about the
# SAME image — get different expert answers.
# =============================================

image = Image.open("living_room.jpg")

questions = [
    # Interior designer perspective
    "As an interior designer, what style is this room? What 3 changes would improve it?",
    
    # Real estate agent perspective
    "As a real estate agent, describe this room for a property listing in 2 sentences.",
    
    # Safety inspector perspective
    "As a safety inspector, identify any potential hazards or issues in this room.",
    
    # Cost estimator perspective
    "Estimate the total cost of furnishing this room in India (INR). Break down by item.",
]

for q in questions:
    print(f"\nQ: {q[:60]}...")
    response = model.generate_content([q, image])
    print(f"A: {response.text[:200]}...\n")
    print("─" * 50)

print("""
Same image, 4 completely different expert analyses.
The AI adapts its "lens" based on the role in your prompt.
This is role-based visual analysis — Module 3.1 meets vision!
""")


> **What just happened?** We sent the same living room photo to Gemini 4 times with 4 different expert roles. The AI produced 4 completely different analyses: an interior design critique, a real estate listing, a safety inspection report, and a cost estimate — all from the same image. The text prompt controls WHAT the AI looks for; the image provides WHAT it looks at. This is the power of multimodal: vision + language reasoning combined.


### Step 2 · Document AI — Extract Data from Photos of Documents

Invoices, receipts, ID cards, handwritten notes — extract structured data from any document image.

**`part2_document_extraction.py`** · _Block 4 of 10_

DOCUMENT AI: Extract structured data from — photos of invoices, receipts, forms.


In [ ]:
# =============================================
# DOCUMENT AI: Extract structured data from
# photos of invoices, receipts, forms.
# =============================================

from pydantic import BaseModel, Field
from typing import Optional
import json, re

# ── Schema for an invoice ──
class LineItem(BaseModel):
    description: str
    quantity: int
    unit_price: float
    total: float

class Invoice(BaseModel):
    vendor_name: str
    invoice_number: Optional[str] = None
    date: str = Field(description="Date in YYYY-MM-DD format")
    items: list[LineItem]
    subtotal: float
    tax: float
    total: float
    currency: str = "INR"

# ── Extract from a photo of an invoice ──
invoice_image = Image.open("invoice_photo.jpg")

prompt = f"""Extract ALL information from this invoice image into structured JSON.

Return ONLY valid JSON matching this schema (no markdown, no extra text):
{json.dumps(Invoice.model_json_schema(), indent=2)}

Rules:
- Extract EVERY line item visible in the invoice
- Convert the date to YYYY-MM-DD format
- If a field is not visible, use null
- Currency should be INR unless another currency is visible"""

response = model.generate_content([prompt, invoice_image])

# Parse and validate
clean = re.sub(r"```json?\s*", "", response.text).replace("```", "").strip()
try:
    invoice = Invoice.model_validate_json(clean)
    print("✅ Invoice extracted and validated!\n")
    print(f"  Vendor:  {invoice.vendor_name}")
    print(f"  Date:    {invoice.date}")
    print(f"  Invoice: {invoice.invoice_number}")
    print(f"  Items:")
    for item in invoice.items:
        print(f"    - {item.description}: {item.quantity}x ₹{item.unit_price} = ₹{item.total}")
    print(f"  Subtotal: ₹{invoice.subtotal:,.2f}")
    print(f"  Tax:      ₹{invoice.tax:,.2f}")
    print(f"  Total:    ₹{invoice.total:,.2f}")
except Exception as e:
    print(f"❌ Extraction failed: {e}")


**`part2_batch_receipts.py`** · _Block 5 of 10_

BATCH RECEIPT PROCESSING — Process a folder of receipt photos → CSV


In [ ]:
# =============================================
# BATCH RECEIPT PROCESSING
# Process a folder of receipt photos → CSV
# =============================================

import csv
from pathlib import Path

class ReceiptProcessor:
    """Extract data from receipt photos into structured records."""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0.1})
    
    def extract(self, image_path: str) -> dict:
        image = Image.open(image_path)
        
        prompt = """Extract from this receipt/bill:
Return ONLY valid JSON:
{
    "store": "store name",
    "date": "YYYY-MM-DD",
    "items": [{"name": "item", "price": number}],
    "total": number,
    "payment_method": "cash/card/upi/unknown"
}"""
        
        response = self.model.generate_content([prompt, image])
        clean = re.sub(r"```json?\s*", "", response.text).replace("```", "").strip()
        return json.loads(clean)
    
    def process_folder(self, folder: str, output_csv: str):
        """Process all receipt images in a folder → CSV file."""
        records = []
        
        for img_path in sorted(Path(folder).glob("*")):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                print(f"  Processing: {img_path.name}...")
                try:
                    data = self.extract(str(img_path))
                    data["source_file"] = img_path.name
                    records.append(data)
                    print(f"    ✅ {data['store']} — ₹{data['total']}")
                except Exception as e:
                    print(f"    ❌ Failed: {e}")
        
        # Write CSV
        with open(output_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["File", "Store", "Date", "Total", "Payment"])
            for r in records:
                writer.writerow([r.get("source_file"), r.get("store"), r.get("date"),
                                 r.get("total"), r.get("payment_method")])
        
        print(f"\n{len(records)} receipts → {output_csv}")
        total_spent = sum(r.get("total", 0) for r in records)
        print(f"Total spending: ₹{total_spent:,.2f}")

# ── Run it ──
processor = ReceiptProcessor()
processor.process_folder("./my_receipts", "expenses.csv")


> **What just happened?** Two document AI workflows: (1) Invoice extraction — a photo of an invoice becomes a validated Pydantic object with vendor name, line items, tax, and total, all in structured JSON. (2) Batch receipt processing — a folder of receipt photos gets processed into a CSV expense report with store names, dates, and totals. The data entry that took a clerk 5 minutes per document now takes 3 seconds. This is how fintech and accounting companies automate document processing.


### Step 3 · Multi-Image Reasoning — Compare, Contrast, Analyze

Send multiple images in one call. The AI understands relationships between them.

**`part3_multi_image.py`** · _Block 6 of 10_

MULTI-IMAGE: Send 2+ images, reason across them


In [ ]:
# =============================================
# MULTI-IMAGE: Send 2+ images, reason across them
# =============================================

# ── Comparison: Before vs After ──
before = Image.open("room_before.jpg")
after = Image.open("room_after.jpg")

response = model.generate_content([
    "These are before and after photos of a room renovation. "
    "First image is BEFORE, second is AFTER. "
    "Analyze: What changed? What improved? What could be done better? "
    "Rate the renovation out of 10.",
    before, after
])
print("Before/After analysis:")
print(response.text)

# ── Product comparison ──
print("\n" + "─" * 50 + "\n")

phone_a = Image.open("phone_a.jpg")
phone_b = Image.open("phone_b.jpg")

response = model.generate_content([
    "Compare these two phones from their photos. "
    "Identify: brand, approximate model, screen size, camera setup. "
    "Which looks more premium? Which would you recommend for a college "
    "student in India? Return your analysis as a comparison table.",
    phone_a, phone_b
])
print("Phone comparison:")
print(response.text)

# ── Spot the difference ──
print("\n" + "─" * 50 + "\n")

# Send multiple photos of a construction site over time
day1 = Image.open("site_day1.jpg")
day30 = Image.open("site_day30.jpg")
day60 = Image.open("site_day60.jpg")

response = model.generate_content([
    "These 3 images show a construction site at Day 1, Day 30, and Day 60. "
    "Analyze the progress: What was built in each phase? "
    "Estimate the completion percentage at each stage. "
    "Are there any visible quality or safety concerns?",
    day1, day30, day60
])
print("Construction progress:")
print(response.text)


> **What just happened?** We sent multiple images in a single API call and the AI reasoned across them: before/after renovation analysis, side-by-side phone comparison, and 3-stage construction progress tracking. The model doesn't just describe each image separately — it understands relationships : what changed, what improved, what's different. This enables use cases like quality inspection, progress monitoring, and visual A/B testing.


### Step 4 · Video Understanding — Watch and Analyze Videos

Gemini can process entire videos — summarize, extract events, and answer questions.

**`part4_video_understanding.py`** · _Block 7 of 10_

VIDEO UNDERSTANDING: Upload and analyze videos — Gemini processes frames + audio together.


In [ ]:
# =============================================
# VIDEO UNDERSTANDING: Upload and analyze videos
# Gemini processes frames + audio together.
# =============================================

import google.generativeai as genai
import time

# ── Upload a video file ──
print("Uploading video...")
video_file = genai.upload_file("product_demo.mp4")

# Wait for processing
while video_file.state.name == "PROCESSING":
    time.sleep(5)
    video_file = genai.get_file(video_file.name)
print(f"Video ready: {video_file.state.name}")

model = genai.GenerativeModel("gemini-2.0-flash")

# ── Summarize the video ──
response = model.generate_content([
    "Watch this video and provide: "
    "1. A 3-sentence summary. "
    "2. Key timestamps with what happens at each. "
    "3. The main product/topic discussed. "
    "4. The target audience.",
    video_file
])
print("Video analysis:")
print(response.text)

# ── Ask specific questions about the video ──
questions = [
    "What are the 3 main features demonstrated in this video?",
    "At what timestamp does the pricing discussion begin?",
    "Summarize this video for a tweet (under 280 characters).",
    "What competitor products are mentioned, if any?",
]

for q in questions:
    response = model.generate_content([q, video_file])
    print(f"\nQ: {q}")
    print(f"A: {response.text[:200]}")


> **What just happened?** We uploaded a video and asked Gemini to: summarize it, extract key timestamps, identify the topic, and answer specific questions like "at what timestamp does pricing begin?" The model processes video frames + audio together, understanding both what's shown and what's said. This enables automated meeting summaries, content moderation, lecture indexing, and video search.


### Step 5 · Project: Smart Product Catalog Generator

Take a photo of any product → get a complete e-commerce listing with structured data.

**`project_product_catalog.py`** · _Block 8 of 10_

PROJECT: Smart Product Catalog Generator — Photo → complete e-commerce listing with


In [ ]:
# =============================================
# PROJECT: Smart Product Catalog Generator
# Photo → complete e-commerce listing with
# title, description, specs, price estimate,
# SEO keywords, and target audience.
# =============================================

from pydantic import BaseModel, Field
from typing import Optional

class ProductSpecs(BaseModel):
    material: Optional[str] = None
    dimensions: Optional[str] = None
    color: str
    weight: Optional[str] = None

class ProductListing(BaseModel):
    title: str = Field(max_length=100, description="SEO-friendly product title")
    category: str
    description: str = Field(max_length=500, description="Compelling product description")
    specs: ProductSpecs
    estimated_price_inr: str = Field(description="Price range in INR")
    target_audience: str
    seo_keywords: list[str] = Field(min_length=3, max_length=8)
    selling_points: list[str] = Field(min_length=3, max_length=5)
    condition: str = Field(description="new/used/refurbished based on appearance")

class ProductCatalogGenerator:
    """Turn product photos into complete e-commerce listings."""
    
    def __init__(self):
        self.model = genai.GenerativeModel(
            "gemini-2.0-flash",
            system_instruction="""You are an expert e-commerce product cataloger.
You analyze product photos and generate complete, compelling listings.
You estimate prices based on the Indian market (2026 prices).
Always return valid JSON matching the requested schema.""",
            generation_config={"temperature": 0.2}
        )
    
    def generate_listing(self, image_path: str) -> ProductListing:
        """Generate a complete product listing from a photo."""
        image = Image.open(image_path)
        
        prompt = f"""Analyze this product photo and create a complete e-commerce listing.

Return ONLY valid JSON matching this schema:
{json.dumps(ProductListing.model_json_schema(), indent=2)}

Make the title SEO-friendly. Make the description compelling.
Estimate price for the Indian market. List the top selling points."""
        
        response = self.model.generate_content([prompt, image])
        clean = re.sub(r"```json?\s*", "", response.text).replace("```", "").strip()
        return ProductListing.model_validate_json(clean)
    
    def compare_products(self, image_paths: list[str]) -> str:
        """Compare multiple products from their photos."""
        images = [Image.open(p) for p in image_paths]
        
        prompt = f"""Compare these {len(images)} products from their photos.
For each product: identify it, estimate price, rate quality (1-10).
Then recommend which offers the best value for money.
Format as a comparison table."""
        
        return self.model.generate_content([prompt] + images).text

# ── Generate listings ──
catalog = ProductCatalogGenerator()

product_images = ["shoe.jpg", "headphones.jpg", "backpack.jpg"]

for img_path in product_images:
    print(f"\n{'═'*50}")
    print(f"Processing: {img_path}")
    print(f"{'═'*50}")
    
    try:
        listing = catalog.generate_listing(img_path)
        print(f"  Title:    {listing.title}")
        print(f"  Category: {listing.category}")
        print(f"  Price:    {listing.estimated_price_inr}")
        print(f"  Color:    {listing.specs.color}")
        print(f"  Keywords: {', '.join(listing.seo_keywords)}")
        print(f"  Selling points:")
        for sp in listing.selling_points:
            print(f"    • {sp}")
        print(f"  Description: {listing.description[:150]}...")
    except Exception as e:
        print(f"  ❌ Error: {e}")

# Compare products
print(f"\n{'═'*50}")
print("Product Comparison")
print(f"{'═'*50}")
comparison = catalog.compare_products(product_images)
print(comparison)

print("""
What this enables:
  • Sellers: snap a photo → get a complete listing in seconds
  • Marketplaces: auto-catalog millions of products
  • Resellers: photo → price estimate → listing → sell
  • Quality control: photo → condition assessment → grading
  
This is how Flipkart and Amazon process seller uploads.
""")


> **What just happened?** We built a complete product catalog generator: take a photo of any product → get an SEO-optimized title, compelling description, specifications, estimated price for the Indian market, target audience, keywords, and selling points — all validated with Pydantic. The comparison feature takes multiple product photos and generates a side-by-side table with value recommendations. This is a real production pipeline used by e-commerce platforms to auto-catalog seller inventory.


**`llava_local.py`** · _Block 9 of 10_

LLaVA: Open-source multimodal model — Runs on your own GPU — no API key needed!


In [ ]:
# =============================================
# LLaVA: Open-source multimodal model
# Runs on your own GPU — no API key needed!
# pip install transformers torch accelerate
# =============================================

from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
from PIL import Image
import torch

# Load model (downloads ~15 GB first time)
processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")
model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-mistral-7b-hf",
    torch_dtype=torch.float16,
).to("cuda")

# Use it — same concept as Gemini but runs locally
image = Image.open("product.jpg")

prompt = "[INST] <image>\nDescribe this product and estimate its price in India. [/INST]"

inputs = processor(prompt, image, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=300)
result = processor.decode(output[0], skip_special_tokens=True)

print(result)

print("""
LLaVA vs Gemini:
  LLaVA: free, local, private, customizable — needs GPU (8+ GB VRAM)
  Gemini: better quality, no GPU needed, supports video — costs per token
  
For most projects: start with Gemini API, switch to LLaVA when
you need privacy, scale, or customization.
""")


### Step 6 · How VLMs Work — Vision Encoder + Projector + LLM

Three components. The projector bridges the modality gap. LLaVA proved simplicity wins.

**`vlm_architecture.py`** · _Block 10 of 10_

═══════ VLM ARCHITECTURE: 3 COMPONENTS ═══════


In [ ]:
# ═══════ VLM ARCHITECTURE: 3 COMPONENTS ═══════

# 1. VISION ENCODER — extracts visual features
#    CLIP ViT-L/14 (LLaVA 1.5) → 768-dim patch embeddings
#    SigLIP-SO400M (LLaVA-OneVision, PaliGemma) → 1152-dim
#    InternViT-6B (InternVL 3.5) → largest open vision encoder

# 2. PROJECTOR — maps vision → LLM token space
#    MLP (LLaVA): 2-layer MLP, simplest, works great
#    Q-Former (BLIP-2): learnable queries, more complex
#    Cross-attention (Flamingo): interleave vision at each layer

# 3. LLM — reasons about visual + text tokens
#    Gemma 2 (PaliGemma), Qwen2 (LLaVA-OneVision)
#    InternLM2 (InternVL), Llama 3 (Llama-Vision)

architecture = {
    "LLaVA-1.5":      {"vision": "CLIP ViT-L/14",     "proj": "2-layer MLP",  "llm": "Vicuna-13B"},
    "LLaVA-OneVision": {"vision": "SigLIP-SO400M",   "proj": "MLP",         "llm": "Qwen2 0.5-72B"},
    "InternVL 3.5":    {"vision": "InternViT-6B",    "proj": "QLLaMA",      "llm": "InternLM2.5"},
    "Qwen2.5-VL":      {"vision": "Custom 600M ViT",  "proj": "Native",      "llm": "Qwen2.5 3-72B"},
    "PaliGemma 2":     {"vision": "SigLIP-SO400M",   "proj": "Linear",      "llm": "Gemma 2 3-28B"},
}
for name, arch in architecture.items():
    print(f"{name:20s}: {arch['vision']:20s} → {arch['proj']:12s} → {arch['llm']}")


> **What just happened?** All VLMs follow the same pattern: Vision Encoder → Projector → LLM . The vision encoder (ViT/SigLIP) extracts visual features. The projector maps them into the LLM's token space. The LLM reasons about visual tokens as if they were text. LLaVA's key insight: a simple 2-layer MLP projector works as well as complex bridging mechanisms (Q-Former, cross-attention). Training is two-stage: (1) train projector only (ViT + LLM frozen, 6 hours), (2) fine-tune projector + LLM together. InternVL 3.5 uses the largest open vision encoder (6B params) and needs 10× fewer training tokens as a result.


---

## Wrap-up

You've walked through all 10 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
